# Artisan

In [ ]:
from artisan.operations.curator import Filter, IngestData, Merge
from artisan.operations.examples import (
    DataGenerator,
    DataTransformer,
    MetricCalculator,
)
from artisan.orchestration import PipelineManager
from artisan.utils import find_project_root, tutorial_setup
from artisan.visualization import (
    build_macro_graph,
    build_micro_graph,
    inspect_metrics,
    inspect_pipeline,
)

PROJECT_ROOT = find_project_root()
SOURCE_FILES = sorted((PROJECT_ROOT / "tests" / "fixtures" / "csv").glob("*.csv"))[:2]

env = tutorial_setup("demo")
DELTA_ROOT = env.delta_root

pipeline = PipelineManager.create(
    name="demo",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)
output = pipeline.output

## Build a pipeline

In [ ]:
# Two data sources: external files and synthetic generation
pipeline.run(operation=IngestData, name="ingest", inputs=[str(f) for f in SOURCE_FILES])
pipeline.run(operation=DataGenerator, name="generate", params={"count": 2, "seed": 44})

# Merge into a single stream — output() wires steps together by name and role
pipeline.run(
    operation=Merge,
    name="merge",
    inputs={
        "branch_a": output("ingest", "data"),
        "branch_b": output("generate", "datasets"),
    },
)

In [ ]:
# Transform, score, and filter
pipeline.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("merge", "merged")},
    params={"scale_factor": 0.5, "variants": 1, "seed": 100},
)
pipeline.run(
    operation=MetricCalculator,
    name="metrics",
    inputs={"dataset": output("transform", "dataset")},
)

# Filter auto-discovers metrics via provenance — no explicit link needed
pipeline.run(
    operation=Filter,
    name="filter",
    inputs={"passthrough": output("transform", "dataset")},
    params={"criteria": [{"metric": "distribution.median", "operator": "gt", "value": 0.15}]},
)

pipeline.finalize()

## Inspect results

In [ ]:
inspect_pipeline(DELTA_ROOT)

In [ ]:
inspect_metrics(DELTA_ROOT, step_number=4).sort("distribution.median", descending=True)

In [ ]:
build_macro_graph(DELTA_ROOT)

In [ ]:
build_micro_graph(DELTA_ROOT)